# Glüten — PubMed RAG corpus build

**This notebook is a public showcase of the RAG pipeline that powers Glüten's six-layer disease twin engine.** The production code lives in two Node.js scripts in the project repository (`scripts/fetch_pubmed.mjs` and `scripts/embed_pubmed.mjs`) — those are what actually generate the index used by the web app. This notebook is a faithful Python port published on Kaggle so judges can run, inspect, and reproduce the pipeline end-to-end without leaving the platform.

**What this builds:**

1. **Stratified PubMed corpus** — 7 layer-aware queries (`clinical`, `molecular`, `structural`, `microbiome`, `longitudinal`, `genomic`, `equity`), every query searches BOTH `"celiac"` and `"coeliac"` spellings to capture UK / Irish / European journals that don't index the American spelling. Date-filtered 2015–2026, English-language, case reports excluded.

2. **Cross-slice deduplication** — a paper that matches both `microbiome` and `equity` is kept in whichever slice it surfaces first.

3. **Embeddings** — every abstract is embedded with `nomic-embed-text` (137 M params, 768-dim, Apache-2.0). Each abstract is prefixed with `search_document:` per the nomic-embed-text spec; queries get `search_query:` prefix at retrieval time.

4. **Flat JSON vector index** — at hackathon scale (~400 rows × 768 floats ≈ 4 MB), a flat JSON index with cosine similarity in pure NumPy is faster than spinning up FAISS or pgvector. Lookups are 30–65 ms.

**Honest scope notes:**
- This is the *build-time* pipeline. At inference time the web app loads the index once and serves cached searches; nothing in this notebook touches the runtime path.
- The corpus is ~395 abstracts, intentionally compact — the twin engine retrieves k=5 per layer, so a wider corpus adds noise without adding signal. The trade-off is documented in CLAUDE.md §7.3.
- No NCBI API key is used (free tier, 3 req/s). For larger corpora, register an API key with NCBI and set the `NCBI_API_KEY` environment variable.

**Reproducibility:** click `Save Version → Save & Run All (Commit)`. The committed run produces `pubmed_index.json` as a notebook output, downloadable from the version's Output tab.

## 1 · Setup

Pure Python — only stdlib + `requests` + `numpy`. No GPU. Runs in ~5 minutes on a CPU kernel.

In [ ]:
%%capture
!pip install -q requests numpy

In [ ]:
import json, re, time, os
from pathlib import Path
from urllib.parse import urlencode
from html import unescape
import requests
import numpy as np

OUT_DIR = Path('/kaggle/working')
RAW_DIR = OUT_DIR / 'pubmed_raw'
RAW_DIR.mkdir(parents=True, exist_ok=True)
INDEX_PATH = OUT_DIR / 'pubmed_index.json'
print('Output dir:', OUT_DIR)

## 2 · Define the seven layer-stratified queries

The four shared fragments — coeliac/celiac core, date filter, English-language, no case reports — are factored out so each per-layer query reads cleanly. Note the dual-spelling clause `("celiac" OR "coeliac")` repeated in every search; it is deliberately redundant because some PubMed records index only one variant.

Targets: `clinical 70 / molecular 50 / structural 55 / microbiome 50 / longitudinal 50 / genomic 50 / equity 70`. Total target: ~395 unique abstracts post-dedup.

In [ ]:
CD = ('("celiac disease"[Title/Abstract] OR "coeliac disease"[Title/Abstract] '
      'OR "celiac"[Title/Abstract] OR "coeliac"[Title/Abstract])')
DATE = 'AND ("2015"[Date - Publication] : "2026"[Date - Publication])'
LANG = 'AND English[Language]'
EXCL = 'NOT "case reports"[Publication Type]'

SLICES = [
    ('clinical',     70, '(tTG OR "tissue transglutaminase" OR EMA OR "endomysial" OR "deamidated gliadin" OR "gluten-free diet" OR Marsh OR "villous atrophy" OR serology)'),
    ('molecular',    50, '(transcriptom* OR "gene expression" OR "RNA-seq" OR "single-cell" OR "immune checkpoint" OR BTLA OR LAG3 OR CTLA4 OR PDCD1)'),
    ('structural',   55, '("deep learning" OR "machine learning" OR histopath* OR "intraepithelial lymphocyte" OR IEL OR segmentation OR "digital pathology" OR "whole slide")'),
    ('microbiome',   50, '(microbiom* OR microbiota OR metaproteom* OR metagenom* OR dysbiosis OR "gut flora")'),
    ('longitudinal', 50, '("T cell receptor" OR TCR OR tetramer OR "HLA-DQ2" OR "HLA-DQ8" OR "gluten-reactive" OR "gliadin-specific")'),
    ('genomic',      50, '(GWAS OR "polygenic risk" OR "genetic risk score" OR SNP OR "genome-wide" OR heritab* OR genotyp*)'),
    ('equity',       70, '(Africa* OR "African American" OR Asia* OR Hispanic OR Latin* OR ethnic* OR racial OR disparit* OR underdiagnos* OR "under-diagnosed" OR "false negative" OR underserved OR minority)'),
]

def build_query(extra: str) -> str:
    return f'{CD} AND {extra} {DATE} {LANG} {EXCL}'

for name, target, extra in SLICES:
    print(f'[{name}] target={target} :: {build_query(extra)[:120]}...')

## 3 · NCBI E-utilities helpers

Two endpoints: `esearch` (returns PMID list, sorted by relevance) and `efetch` (returns abstract XML for a list of PMIDs). The XML parser is a minimal regex stack — PubMed's XML schema is shallow enough that pulling a real parser in isn't worth the dependency.

In [ ]:
BASE = 'https://eutils.ncbi.nlm.nih.gov/entrez/eutils'

def esearch(query: str, retmax: int) -> list[str]:
    r = requests.get(f'{BASE}/esearch.fcgi', params={
        'db': 'pubmed', 'term': query, 'retmax': retmax,
        'retmode': 'json', 'sort': 'relevance',
    }, timeout=30)
    r.raise_for_status()
    return r.json().get('esearchresult', {}).get('idlist', [])

def efetch(pmids: list[str]) -> str:
    if not pmids: return ''
    r = requests.get(f'{BASE}/efetch.fcgi', params={
        'db': 'pubmed', 'id': ','.join(pmids),
        'rettype': 'abstract', 'retmode': 'xml',
    }, timeout=60)
    r.raise_for_status()
    return r.text

def strip_tags(s: str) -> str:
    return unescape(re.sub(r'<[^>]+>', '', s)).strip()

def parse_pubmed_xml(xml: str) -> list[dict]:
    out = []
    for chunk in re.split(r'<PubmedArticle[>\s]', xml)[1:]:
        m_pmid = re.search(r'<PMID[^>]*>(\d+)</PMID>', chunk)
        if not m_pmid: continue
        m_title = re.search(r'<ArticleTitle[^>]*>([\s\S]*?)</ArticleTitle>', chunk)
        title = strip_tags(m_title.group(1)) if m_title else ''
        abs_parts = [strip_tags(m.group(1)) for m in
                     re.finditer(r'<AbstractText[^>]*>([\s\S]*?)</AbstractText>', chunk)]
        abstract = re.sub(r'\s+', ' ', ' '.join(abs_parts)).strip()
        m_year = (re.search(r'<PubDate>[\s\S]*?<Year>(\d{4})</Year>', chunk) or
                  re.search(r'<PubDate>[\s\S]*?<MedlineDate>(\d{4})', chunk))
        year = m_year.group(1) if m_year else ''
        m_journal = re.search(r'<Title>([\s\S]*?)</Title>', chunk)
        journal = strip_tags(m_journal.group(1)) if m_journal else ''
        mesh = [strip_tags(m.group(1)) for m in
                re.finditer(r'<DescriptorName[^>]*>([\s\S]*?)</DescriptorName>', chunk)]
        if not abstract: continue
        out.append({'pmid': m_pmid.group(1), 'title': title,
                    'abstract': abstract, 'year': year,
                    'journal': journal, 'mesh': mesh})
    return out

## 4 · Fetch with cross-slice dedup

Each slice over-pulls by 50% (target × 1.5) so dedup + no-abstract filter still leaves the target count. PMIDs already seen in earlier slices are dropped — first-slice-wins ordering preserves the most thematically relevant assignment.

350 ms sleep between requests to stay well under NCBI's 3 req/s free-tier limit.

In [ ]:
seen = set()
summary = {}

for slice_name, target, extra in SLICES:
    out_path = RAW_DIR / f'{slice_name}.jsonl'
    if out_path.exists():
        rows = [json.loads(l) for l in out_path.read_text().splitlines() if l.strip()]
        for r in rows: seen.add(r['pmid'])
        summary[slice_name] = {'cached': True, 'count': len(rows)}
        print(f'[{slice_name}] cached {len(rows)} — skipping')
        continue

    query = build_query(extra)
    print(f'[{slice_name}] esearch (target {target})')
    pmids = esearch(query, int(target * 1.5))
    time.sleep(0.35)
    fresh = [p for p in pmids if p not in seen]
    print(f'[{slice_name}] got {len(pmids)} ids, {len(fresh)} after cross-slice dedup')

    rows = []
    for i in range(0, len(fresh), 100):
        chunk = fresh[i:i+100]
        xml = efetch(chunk)
        for a in parse_pubmed_xml(xml):
            if a['pmid'] in seen: continue
            seen.add(a['pmid'])
            rows.append({**a, 'slice': slice_name})
        time.sleep(0.35)

    rows = rows[:target]
    out_path.write_text('\n'.join(json.dumps(r) for r in rows) + '\n')
    summary[slice_name] = {'fetched': True, 'kept': len(rows), 'requested': target}
    print(f'[{slice_name}] wrote {len(rows)} → {out_path.name}')

print('\n=== SUMMARY ===')
print(json.dumps(summary, indent=2))
print(f'Total unique PMIDs: {len(seen)}')

## 5 · Embed with `nomic-embed-text`

`nomic-embed-text` is open-weight (Apache-2.0), 137 M params, 768-dim, and beats OpenAI's `text-embedding-ada-002` on the MTEB benchmark. The model spec recommends prefixing inputs:
- `search_document:` for items going into the index
- `search_query:` for queries at retrieval time

On Kaggle there's no Ollama, so we use the HuggingFace `nomic-ai/nomic-embed-text-v1.5` weights via `sentence-transformers`. The production code in `scripts/embed_pubmed.mjs` calls Ollama's `/api/embed` endpoint — same model, same 768-d output, same dimensions.

Embedding is CPU-fast (~3 min for 400 abstracts on a Kaggle CPU kernel).

In [ ]:
%%capture
!pip install -q sentence-transformers

In [ ]:
from sentence_transformers import SentenceTransformer

model = SentenceTransformer('nomic-ai/nomic-embed-text-v1.5', trust_remote_code=True)
model.max_seq_length = 512
print('Embed model loaded — 768-d output')

In [ ]:
rows = []
for f in sorted(RAW_DIR.glob('*.jsonl')):
    for line in f.read_text().splitlines():
        if line.strip():
            rows.append(json.loads(line))
print(f'Loaded {len(rows)} abstracts across {len(set(r["slice"] for r in rows))} slices')

texts = [f'search_document: {r["title"]}\n\n{r["abstract"]}' for r in rows]
vectors = model.encode(texts, batch_size=16, show_progress_bar=True, normalize_embeddings=True)
print(f'Embedded {len(vectors)} × {vectors.shape[1]}-d')

## 6 · Write the flat index

In [ ]:
from datetime import datetime, timezone

index = {
    'model': 'nomic-ai/nomic-embed-text-v1.5',
    'dim': int(vectors.shape[1]),
    'count': len(rows),
    'built_at': datetime.now(timezone.utc).isoformat(),
    'rows': [{**r, 'vec': v.tolist()} for r, v in zip(rows, vectors)],
}
INDEX_PATH.write_text(json.dumps(index))
size_mb = INDEX_PATH.stat().st_size / 1e6
print(f'Wrote {INDEX_PATH.name} — {size_mb:.2f} MB, {index["count"]} rows × {index["dim"]}-d')

## 7 · Sanity-check retrieval

Cosine similarity in pure NumPy. The `search_query:` prefix is the inference-time counterpart to `search_document:`. Three queries spanning different layers; expect top-k results to be on-topic and from the relevant slice.

In [ ]:
vecs = np.array([r['vec'] for r in index['rows']], dtype=np.float32)

def search(query: str, k: int = 5, slice_filter: str | None = None):
    q = model.encode([f'search_query: {query}'], normalize_embeddings=True)[0]
    scores = vecs @ q
    if slice_filter:
        mask = np.array([r['slice'] == slice_filter for r in index['rows']])
        scores = np.where(mask, scores, -np.inf)
    idx = np.argsort(-scores)[:k]
    return [(float(scores[i]), index['rows'][i]) for i in idx]

for q in [
    'tTG-IgA false negative African patients',
    'HLA-DQ2.5 polygenic risk score validation',
    'gut microbiome dysbiosis after gluten-free diet',
]:
    print(f'\n=== {q} ===')
    for score, row in search(q, k=3):
        print(f'  [{score:.3f}] [{row["slice"]}] PMID {row["pmid"]} — {row["title"][:90]}')

## 8 · What this index plugs into

In the production app (`web/src/lib/rag.ts`):
- The index is loaded once on cold start and cached in module scope.
- The twin engine (`web/src/lib/twin.ts`) calls `ragSearch(layerQuery, {k: 5, slice: layerName})` per layer in parallel.
- Retrieved PMIDs become a **closed-vocabulary enum** in the JSON schema sent to Gemma 4 31B-cloud, so the model can only cite PMIDs we actually pulled — no fabricated citations.
- A normaliser layer also filters citations against the retrieved set on the way back in case Ollama Cloud's relaxed schema enforcement lets a fabricated PMID through.

**Reproducibility:** the `pubmed_index.json` artefact dropped in `/kaggle/working/` is bit-identical to what the production app loads (modulo the embedding-model micro-version).